# Does RoPE Prevent or Degrade Retrieval Heads?
## A Mechanistic Analysis Across Model Families

**Google Colab L4 GPU notebook** — run cells top to bottom.  
Results are persisted to Google Drive so long-running analyses can be resumed across sessions.

**Estimated total runtime**: ~15–25 hours (GPU-dependent).  
Layer B (OLMo checkpoints) is optional and adds ~12–25 hours.


In [ ]:
# ── Cell 0: GPU check and Google Drive mount ─────────────────────────────────
# Expected: NVIDIA L4 (24 GB). If you see a different GPU, change runtime.
import subprocess, os
print(subprocess.check_output('nvidia-smi', shell=True).decode())

from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/rope_retrieval_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

In [ ]:
%%bash
# ── Cell 1: Dependency installation (~3 min) ─────────────────────────────────
pip install -q transformers==4.47.0 bitsandbytes accelerate datasets
pip install -q scipy scikit-learn matplotlib seaborn pandas huggingface_hub tqdm pyyaml
echo 'Installation complete.'

In [ ]:
# ── Cell 2a: Tokens + clone repository ───────────────────────────────────────
# Tokenlerini aşağıya yapıştır, sonra hücreyi çalıştır.
# Repo PRIVATE olduğu için klonlamak GitHub token gerektirir.
#   GitHub token: https://github.com/settings/tokens   ('repo' scope ile)
#   HF token:     https://huggingface.co/settings/tokens
import os, subprocess

GITHUB_TOKEN = "ghp_PASTE_YOUR_TOKEN_HERE"   # ← buraya yapıştır (private repo için zorunlu)
HF_TOKEN     = "hf_PASTE_YOUR_TOKEN_HERE"    # ← buraya yapıştır (gated modeller için)

if GITHUB_TOKEN and "PASTE" not in GITHUB_TOKEN:
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
if HF_TOKEN and "PASTE" not in HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

REPO_OWNER = "CengizhanBayram"
REPO_NAME  = "Does-RoPE-Prevent-or-Degrade-Retrieval-Heads-A-Mechanistic-Analysis-Across-Model-Families"
REPO_DIR   = "/content/rope-retrieval-heads"

_tok = os.environ.get('GITHUB_TOKEN')
public_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
clone_url  = f"https://{_tok}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if _tok else public_url

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True)
    # Token'ı remote config'te bırakmamak için temiz URL'e geri al.
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", public_url], check=True)
else:
    print("Repo zaten klonlanmış, güncelleniyor …")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

print("Klonlama tamam →", REPO_DIR)

In [ ]:
# ── Cell 2b: Imports and config ──────────────────────────────────────────────
import sys, os, json, gc, shutil
sys.path.insert(0, '/content/rope-retrieval-heads')

from src.retrieval_head_detector import RetrievalHeadDetector
from src.dimension_utility import DimensionUtilityAnalyzer
from src.olmo_checkpoint_loader import OLMoCheckpointLoader
from src.activation_patching import ActivationPatcher
from src.niah_evaluator import NIAHEvaluator
from src.visualization import (
    plot_retrieval_vs_utility_scatter,
    plot_dimension_utility_profile,
    plot_olmo_training_dynamics,
    plot_theta_comparison_heatmaps,
    plot_activation_patching_results,
    plot_niah_comparison,
)
import yaml, numpy as np, torch
from pathlib import Path
from datetime import datetime

with open('/content/rope-retrieval-heads/configs/config.yaml') as f:
    config = yaml.safe_load(f)

# Override results dir to Google Drive
config['output']['results_dir'] = RESULTS_DIR

print('Config loaded. Models available:', list(config['models'].keys()))
print(f'torch version: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Cell 3: Authentication — HuggingFace + GitHub ────────────────────────────
# Tokenler Cell 2a'da girildi; bu hücre onları kullanarak giriş yapar.
import os
from src.auth_utils import login_huggingface, GitHubClient

# --- HuggingFace (gated modeller: LLaMA-2, LLaMA-3.1) ---
login_huggingface(required=False)

# --- GitHub (opsiyonel: sonuçları repoya yüklemek için) ---
github = None
if os.environ.get('GITHUB_TOKEN'):
    github = GitHubClient(required=False)
    try:
        user = github.whoami()
        print(f"GitHub'a giriş yapıldı: {user['login']}")
    except Exception as exc:
        print(f'GitHub doğrulaması başarısız: {exc}')
        github = None
else:
    print('GitHub token yok — sonuç yükleme kapalı.')

## Layer A — Static Multi-Model Analysis
Estimated time: ~2–3 hours per model, ~6–9 hours total for all three.

In [ ]:
# ── Cell 4: Layer A helper functions ─────────────────────────────────────────
import random, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)-8s  %(message)s')

def set_seeds(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def load_model(model_name, model_cfg):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    hf_id = model_cfg['hf_id']
    tokenizer = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        hf_id, quantization_config=bnb_cfg, device_map='auto', trust_remote_code=True
    )
    model.eval()
    print(f'  VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB')
    return model, tokenizer

def unload_model(model):
    del model; gc.collect(); torch.cuda.empty_cache()

def determine_hypothesis(stats):
    p = stats.get('p_value', float('nan'))
    if p != p: return 'inconclusive'
    if p < 0.05:
        return 'H2' if stats['retrieval_mean_utility'] < stats['non_retrieval_mean_utility'] else 'anomaly'
    return 'inconclusive'

def run_model_analysis(model_name, seed=42):
    """Full Layer-A pipeline for one model. Returns result dict."""
    out_dir = Path(RESULTS_DIR) / 'layer_a' / model_name
    result_path = out_dir / 'results.json'
    if result_path.exists():
        print(f'[{model_name}] Loading cached results …')
        with open(result_path) as f:
            return json.load(f)

    out_dir.mkdir(parents=True, exist_ok=True)
    set_seeds(seed)
    model_cfg = config['models'][model_name]
    niah_cfg = config['niah']

    print(f'[{model_name}] Loading model …')
    model, tokenizer = load_model(model_name, model_cfg)

    print(f'[{model_name}] Generating NIAH samples …')
    detector = RetrievalHeadDetector(
        model, tokenizer, config,
        score_threshold=niah_cfg.get('score_threshold', 0.1), seed=seed
    )
    samples = detector.generate_niah_samples(
        n_samples=niah_cfg['n_samples'],
        context_lengths=niah_cfg['context_lengths'],
        needle_positions=niah_cfg['needle_positions'],
    )

    print(f'[{model_name}] Scoring retrieval heads …')
    scores = detector.score_heads(samples)
    retrieval_heads = detector.get_retrieval_heads(scores)
    np.save(out_dir / 'retrieval_scores.npy', scores)

    print(f'[{model_name}] Computing dimension utility …')
    analyzer = DimensionUtilityAnalyzer(model, config)
    norms = analyzer.compute_query_projection_norms()
    np.save(out_dir / 'norms.npy', norms)
    utility_stats = analyzer.compute_utility_scores(norms, retrieval_heads)
    correlation = analyzer.compute_retrieval_utility_correlation(scores, norms)
    freq_profile = analyzer.dimension_profile_by_frequency(norms, retrieval_heads)

    n_layers, n_heads = scores.shape
    stat_tests = {
        'retrieval_mean_utility': utility_stats['retrieval_mean'],
        'non_retrieval_mean_utility': utility_stats['non_retrieval_mean'],
        't_statistic': utility_stats['t_statistic'],
        'p_value': utility_stats['p_value'],
        'pearson_r': correlation['pearson_r'],
        'pearson_p': correlation['pearson_p'],
        'spearman_rho': correlation['spearman_rho'],
        'spearman_p': correlation['spearman_p'],
    }
    result = {
        'model': model_name,
        'timestamp': datetime.utcnow().isoformat(),
        'seed': seed,
        'retrieval_heads': retrieval_heads,
        'retrieval_scores': scores.tolist(),
        'dimension_utility': {
            'per_head_scalar': utility_stats['per_head_scalar'],
            'frequency_profile': freq_profile,
        },
        'statistical_tests': stat_tests,
        'summary': {
            'n_retrieval_heads': len(retrieval_heads),
            'n_total_heads': n_layers * n_heads,
            'retrieval_fraction': len(retrieval_heads) / (n_layers * n_heads),
            'hypothesis_supported': determine_hypothesis(stat_tests),
        },
    }
    with open(result_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'[{model_name}] Saved → {result_path}')

    unload_model(model)
    return result

print('Layer A helpers defined.')

In [ ]:
# ── Cell 5: LLaMA-3.1-8B (~2–3 hours) ───────────────────────────────────────
llama3_results = run_model_analysis('llama3')
print('\nSummary:', llama3_results['summary'])

In [ ]:
# ── Cell 6: Qwen2.5-7B (~2–3 hours) ─────────────────────────────────────────
qwen_results = run_model_analysis('qwen')
print('\nSummary:', qwen_results['summary'])

In [ ]:
# ── Cell 7: OLMo-2-7B (~2–3 hours) ──────────────────────────────────────────
olmo_results = run_model_analysis('olmo')
print('\nSummary:', olmo_results['summary'])

In [ ]:
# ── Cell 7b: Layer A figures ─────────────────────────────────────────────────
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams['figure.dpi'] = 100  # inline display

all_results = {
    k: v for k, v in [
        ('llama3', llama3_results),
        ('qwen', qwen_results),
        ('olmo', olmo_results),
    ] if v is not None
}

fig_dir = Path(RESULTS_DIR) / 'layer_a' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

plot_retrieval_vs_utility_scatter(all_results, fig_dir / 'figure1_scatter.pdf')
plot_dimension_utility_profile(all_results, fig_dir / 'figure2_frequency_profile.pdf')
print('Layer A figures saved to Drive.')

## Layer B — OLMo-2 Training Dynamics (optional)
Requires ~12 hours with `step_stride=20000` or ~25 hours with `step_stride=10000`.
Each checkpoint result is written to Drive immediately — safe to interrupt and resume.

In [ ]:
# ── Cell 8: Layer B — OLMo checkpoint sweep ───────────────────────────────────
# Set SKIP_LAYER_B = True to skip this section.
SKIP_LAYER_B = False
STEP_STRIDE = 20000  # increase to 20000 for faster run (~25 checkpoints)

if not SKIP_LAYER_B:
    from tqdm.notebook import tqdm as tqdm_nb

    loader = OLMoCheckpointLoader(config)
    filtered = loader.get_filtered_checkpoints(step_stride=STEP_STRIDE)
    print(f'Processing {len(filtered)} checkpoints (stride={STEP_STRIDE}) …')

    checkpoint_results = []

    for revision in tqdm_nb(filtered, desc='OLMo checkpoints'):
        step = loader.get_step_from_revision(revision)
        if step is None:
            continue
        if loader.is_computed(revision, RESULTS_DIR):
            print(f'  Skipping step={step} (cached).')
            out_path = Path(RESULTS_DIR) / 'layer_b' / f'checkpoint_{step}' / 'results.json'
            with open(out_path) as f:
                checkpoint_results.append(json.load(f))
            continue

        print(f'  Loading step={step} …')
        try:
            model, tokenizer = loader.load_checkpoint(revision)
        except Exception as exc:
            print(f'  Failed: {exc}'); continue

        set_seeds(42)
        detector = RetrievalHeadDetector(model, tokenizer, config, seed=42)
        samples = detector.generate_niah_samples(
            n_samples=50,
            context_lengths=config['niah']['context_lengths'],
            needle_positions=config['niah']['needle_positions'],
        )
        scores = detector.score_heads(samples)
        retrieval_heads = detector.get_retrieval_heads(scores)

        analyzer = DimensionUtilityAnalyzer(model, config)
        norms = analyzer.compute_query_projection_norms()
        utility_stats = analyzer.compute_utility_scores(norms, retrieval_heads)
        freq_profile = analyzer.dimension_profile_by_frequency(norms, retrieval_heads)

        n_layers, n_heads_c = scores.shape
        res = {
            'step': step, 'revision': revision,
            'timestamp': datetime.utcnow().isoformat(),
            'retrieval_heads': retrieval_heads,
            'retrieval_scores': scores.tolist(),
            'dimension_utility': {
                'per_head_scalar': utility_stats['per_head_scalar'],
                'frequency_profile': freq_profile,
            },
            'summary': {
                'n_retrieval_heads': len(retrieval_heads),
                'n_total_heads': n_layers * n_heads_c,
            },
        }
        out_dir = Path(RESULTS_DIR) / 'layer_b' / f'checkpoint_{step}'
        out_dir.mkdir(parents=True, exist_ok=True)
        with open(out_dir / 'results.json', 'w') as f:
            json.dump(res, f, indent=2)

        checkpoint_results.append(res)
        loader.clear_checkpoint(model)
        print(f'  step={step}: {len(retrieval_heads)} retrieval heads found.')

    print(f'Layer B complete: {len(checkpoint_results)} checkpoints processed.')
else:
    print('Layer B skipped (SKIP_LAYER_B=True).')
    checkpoint_results = []

In [ ]:
# ── Cell 9: Layer B — Figure 3 ───────────────────────────────────────────────
if len(checkpoint_results) >= 2:
    fig_dir = Path(RESULTS_DIR) / 'layer_b' / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    plot_olmo_training_dynamics(checkpoint_results, fig_dir / 'figure3_dynamics.pdf')
    print('Figure 3 saved.')
else:
    print('Not enough checkpoint results for Figure 3.')

## Layer C — θ Comparison: LLaMA-2 vs LLaMA-3.1
Estimated time: ~4–6 hours.

In [ ]:
# ── Cell 11: LLaMA-2-7B (~2–3 hours) ────────────────────────────────────────
llama2_results = run_model_analysis('llama2')
print('\nSummary:', llama2_results['summary'])

In [ ]:
# ── Cell 12: Layer C — Figures 4 ─────────────────────────────────────────────
fig_dir = Path(RESULTS_DIR) / 'layer_c' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

plot_theta_comparison_heatmaps(
    llama2_results, llama3_results,
    fig_dir / 'figure4_theta_comparison.pdf'
)
plot_dimension_utility_profile(
    {'llama2': llama2_results, 'llama3': llama3_results},
    fig_dir / 'figure2c_theta_freq_profiles.pdf'
)

# H1 assessment
n2 = llama2_results['summary']['n_retrieval_heads']
n3 = llama3_results['summary']['n_retrieval_heads']
print(f'LLaMA-2 (θ=10K): {n2} retrieval heads')
print(f'LLaMA-3 (θ=500K): {n3} retrieval heads')
if n3 < n2:
    print('→ H1 supported: higher θ leads to fewer retrieval heads.')
elif n3 > n2:
    print('→ H1 not supported: higher θ leads to MORE retrieval heads.')
else:
    print('→ Inconclusive: equal retrieval head counts.')

## Layer D — Activation Patching (Causality Test)
Estimated time: ~3–4 hours.

In [ ]:
# ── Cell 13: Layer D — Patching experiment ───────────────────────────────────
# Use the model with the most retrieval heads
best_model = max(
    [('llama3', llama3_results), ('qwen', qwen_results), ('olmo', olmo_results)],
    key=lambda x: x[1]['summary']['n_retrieval_heads']
)[0]
print(f'Using best model for patching: {best_model}')

layer_a_res = {k: v for k, v in [
    ('llama3', llama3_results), ('qwen', qwen_results), ('olmo', olmo_results)
] if k == best_model}[best_model]

retrieval_heads_d = [tuple(h) for h in layer_a_res['retrieval_heads']]
utility_scores_d = layer_a_res['dimension_utility']

# Load norms
norms_path = Path(RESULTS_DIR) / 'layer_a' / best_model / 'norms.npy'
if norms_path.exists():
    utility_scores_d['_norms'] = np.load(norms_path)

set_seeds(42)
model_d, tokenizer_d = load_model(best_model, config['models'][best_model])

detector_d = RetrievalHeadDetector(model_d, tokenizer_d, config, seed=42)
samples_d = detector_d.generate_niah_samples(
    n_samples=50,
    context_lengths=config['niah']['context_lengths'],
    needle_positions=config['niah']['needle_positions'],
)

patcher = ActivationPatcher(model_d, tokenizer_d, config)
patching_results_d = patcher.run_patching_experiment(
    retrieval_heads=retrieval_heads_d,
    utility_scores=utility_scores_d,
    samples=samples_d,
    k_dims=config['activation_patching']['k_dims'],
    n_samples=50,
)
causal_df = patcher.compute_causal_effect(patching_results_d)
print(causal_df[['layer', 'head', 'baseline', 'low_utility', 'random', 'high_utility', 'causal_effect']])

In [ ]:
# ── Cell 14: Layer D — Save and Figure 5 ─────────────────────────────────────
out_dir_d = Path(RESULTS_DIR) / 'layer_d' / best_model
out_dir_d.mkdir(parents=True, exist_ok=True)

patching_serializable = {
    f'layer{l}_head{h}': v for (l, h), v in patching_results_d.items()
}
with open(out_dir_d / 'results.json', 'w') as f:
    json.dump({
        'model': best_model,
        'timestamp': datetime.utcnow().isoformat(),
        'patching_results': patching_serializable,
        'causal_effects': causal_df.to_dict(orient='records'),
        'summary': {
            'mean_causal_effect': float(causal_df['causal_effect'].mean()),
            'hypothesis': 'H2' if causal_df['causal_effect'].mean() < 0 else 'unexpected',
        },
    }, f, indent=2)

fig_dir_d = out_dir_d / 'figures'
fig_dir_d.mkdir(parents=True, exist_ok=True)
plot_activation_patching_results(patching_results_d, fig_dir_d / 'figure5_patching.pdf')
print('Figure 5 saved.')

unload_model(model_d)

In [ ]:
# ── Cell 15: Summary Table ────────────────────────────────────────────────────
import pandas as pd

rows = []
for model_name, res in [
    ('llama3', llama3_results), ('llama2', llama2_results),
    ('qwen', qwen_results), ('olmo', olmo_results)
]:
    if res is None:
        continue
    s = res['summary']
    t = res['statistical_tests']
    ce_val = float(causal_df['causal_effect'].mean()) if model_name == best_model else float('nan')
    rows.append({
        'Model': model_name,
        'θ': config['models'][model_name]['theta'],
        'n_retrieval_heads': s['n_retrieval_heads'],
        'retrieval_fraction': round(s['retrieval_fraction'], 4),
        'retrieval_mean_utility': round(t['retrieval_mean_utility'], 4),
        'non_retrieval_mean_utility': round(t['non_retrieval_mean_utility'], 4),
        't_stat': round(t['t_statistic'], 3),
        'p_value': f"{t['p_value']:.3e}",
        'pearson_r': round(t['pearson_r'], 3),
        'causal_effect': round(ce_val, 4),
        'hypothesis': s['hypothesis_supported'],
    })

df_summary = pd.DataFrame(rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
display(df_summary)

# Save summary CSV
df_summary.to_csv(Path(RESULTS_DIR) / 'summary_table.csv', index=False)
print('Summary table saved to Drive.')

In [ ]:
# ── Cell 16: Export everything to Drive ──────────────────────────────────────
# Results are already written directly to RESULTS_DIR throughout the notebook.
# This cell copies any locally generated figures to Drive as a safety net.

local_results = '/content/rope-retrieval-heads/results'
if os.path.exists(local_results):
    shutil.copytree(local_results, RESULTS_DIR, dirs_exist_ok=True)
    print(f'Local results synced to {RESULTS_DIR}')

print(f'All results available at: {RESULTS_DIR}')
for p in sorted(Path(RESULTS_DIR).rglob('*.json')):
    print(' ', p.relative_to(RESULTS_DIR))